In [1]:
# --- BIBLIOTHÈQUES STANDARDS (Python) ---
import re               # Pour les expressions régulières (supprimer les URLs)
import string           # Pour accéder à la liste complète de ponctuation

# --- NLP : PRÉTRAITEMENT & RACINISATION (NLTK) ---
import nltk             # La base pour la tokenization et les stopwords
from nltk.tokenize import word_tokenize        # Pour découper le texte en mots
from nltk.corpus import stopwords              # Pour filtrer les mots vides (the, is, etc.)
from nltk.stem.porter import PorterStemmer    # Pour le stemming (racinisation)

# --- NLP : ANALYSE AVANCÉE (spaCy & TextBlob) ---
import spacy            # Pour la lemmatisation et le tagging haute précision
from textblob import TextBlob # Pour l'analyse de sentiment et les corrections rapides

# --- OUTILS SPÉCIALISÉS ---
import inflect          # Pour convertir les chiffres (3) en mots (three)
import emoji            # Pour détecter et supprimer les emojis 

# Pour NLTK
nltk.download('punkt')      # Modèle de tokenisation
nltk.download('stopwords')  # Liste des mots vides

# Pour spaCy : téléchargement du modèle anglais (à exécuter une seule fois)
!python -m spacy download en_core_web_sm

[nltk_data] Downloading package punkt to /Users/Licas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/Licas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


zsh:1: command not found: python


In [2]:
import pandas as pd

df = pd.read_csv('SMS_test.csv', encoding='latin-1')



In [3]:


df.isnull().sum()

S. No.          0
Message_body    0
Label           0
dtype: int64

In [4]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   S. No.        125 non-null    int64
 1   Message_body  125 non-null    str  
 2   Label         125 non-null    str  
dtypes: int64(1), str(2)
memory usage: 18.4 KB


,S. No.
count,125.000000
mean,63.000000
std,36.228442
min,1.000000
25%,32.000000
50%,63.000000
75%,94.000000
max,125.000000


In [5]:
df.head(10)

,S. No.,Message_body,Label
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam
1,2,"Loan for any purpose £500 - £75,000. Homeowner...",Spam
2,3,Congrats! Nokia 3650 video camera phone is you...,Spam
3,4,URGENT! Your Mobile number has been awarded wi...,Spam
4,5,Someone has contacted our dating service and e...,Spam
5,6,Send a logo 2 ur lover - 2 names joined by a h...,Spam
6,7,FREE entry into our £250 weekly competition ju...,Spam
7,8,100 dating service cal;l 09064012103 box334sk38ch,Spam
8,9,FREE RINGTONE text FIRST to 87131 for a poly o...,Spam
9,10,4mths half price Orange line rental & latest c...,Spam


In [6]:
messages = df["Message_body"]

messages.head()

0    UpgrdCentre Orange customer, you may now claim...
1    Loan for any purpose £500 - £75,000. Homeowner...
2    Congrats! Nokia 3650 video camera phone is you...
3    URGENT! Your Mobile number has been awarded wi...
4    Someone has contacted our dating service and e...
Name: Message_body, dtype: str

# Text Normalization (Lowercasing, Handling contractions, Spell schecking)


In [7]:
# Lowercasing

# on définit notre fonction lowercase
def text_lower(text):
    return text.str.lower()


# on utilise notre fct
text = text_lower(messages)




In [8]:
print(text)

# on check si TOUTE la colonne est déjà en minuscules
est_propre = text.equals(text.str.lower())
print(f"\nLa colonne est-elle totalement en minuscules ? {est_propre}")



0      upgrdcentre orange customer, you may now claim...
1      loan for any purpose £500 - £75,000. homeowner...
2      congrats! nokia 3650 video camera phone is you...
3      urgent! your mobile number has been awarded wi...
4      someone has contacted our dating service and e...
                             ...                        
120    7 wonders in my world 7th you 6th ur style 5th...
121    try to do something dear. you read something f...
122    sun ah... thk mayb can if dun have anythin on....
123    symptoms when u are in love: "1.u like listeni...
124    great. have a safe trip. dont panic surrender ...
Name: Message_body, Length: 125, dtype: str

La colonne est-elle totalement en minuscules ? True


In [9]:
# dealing with contractions

import contractions

def fix_contractions(text):
    if not isinstance(text, str):
        return text
    
    expanded_text = contractions.fix(text)
    return expanded_text

text = fix_contractions(text)


In [10]:
print(text)

0      upgrdcentre orange customer, you may now claim...
1      loan for any purpose £500 - £75,000. homeowner...
2      congrats! nokia 3650 video camera phone is you...
3      urgent! your mobile number has been awarded wi...
4      someone has contacted our dating service and e...
                             ...                        
120    7 wonders in my world 7th you 6th ur style 5th...
121    try to do something dear. you read something f...
122    sun ah... thk mayb can if dun have anythin on....
123    symptoms when u are in love: "1.u like listeni...
124    great. have a safe trip. dont panic surrender ...
Name: Message_body, Length: 125, dtype: str


In [11]:
def fix_contractions(text_list):
    return [contractions.fix(message) for message in text_list]


text = fix_contractions(text)
print(text[0]) # Devrait afficher "...you may now claim..."

upgrdcentre orange customer, you may now claim your free camera phone upgrade for your loyalty. call now on 0207 153 9153. offer ends 26th july. t&c's apply. opt-out available


In [12]:
# spell checking 
!pip install autocorrect
from autocorrect import Speller


def correct(text_list, lang="en"):
    "Correct spellin using autocorrect for english."
    spell = Speller(lang=lang)
    return [spell(message) for message in text_list]

text = correct(text)


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [13]:
print(text)

["upgrdcentre orange customer, you may now claim your free camera phone upgrade for your loyalty. call now on 0207 153 9153. offer ends 26th july. t&c's apply. opt-out available", "loan for any purpose £500 - £75,000. homeowners + tenants welcome. have you been previously refused? we can still help. call free 0800 1956669 or text back 'help'", 'contrast! nokia 3650 video camera phone is your call 09066382422 calls cost 150ppm ave call 3mins vary from mobiles 16+ close 300603 post bcm4284 lon wc1n3xx', 'urgent! your mobile number has been awarded with a £2000 prize guaranteed. call 09058094455 from land line. claim 3030. valid 12hrs only', 'someone has contacted our dating service and entered your phone because they fancy you! to find out who it is call from a landing 09111032124 . box12n146tf150p', 'send a logo 2 you are lover - 2 names joined by a heart. txt love name1 name2 mono eg love adam eve 07123456789 to 87077 yahoo! box36504w45wq xeno 4 no ads 150p', 'free entry into our £250 

In [14]:
# On sauvegarde le résultat de la Normalisation (minuscule + contractions + correction)
# dans une nouvelle colonne AVANT de passer au nettoyage du bruit (Noise Removal).
df["Case Normalization"] = list(text)
df[["Message_body", "Case Normalization"]].head()

,Message_body,Case Normalization
0,"UpgrdCentre Orange customer, you may now claim...","upgrdcentre orange customer, you may now claim..."
1,"Loan for any purpose £500 - £75,000. Homeowner...","loan for any purpose £500 - £75,000. homeowner..."
2,Congrats! Nokia 3650 video camera phone is you...,contrast! nokia 3650 video camera phone is you...
3,URGENT! Your Mobile number has been awarded wi...,urgent! your mobile number has been awarded wi...
4,Someone has contacted our dating service and e...,someone has contacted our dating service and e...


# NOISE REMOVAL

In [15]:
# Noise Removal (Removing numbers/digits, Punctuation & Special Characters,
# Handling double whitespace from text, and Removal of URLs)

# --- Removal of URLs ---
import re

combined_url_pattern = re.compile(r'https?://\S+|www\.\S+')

def remove_urls(text_list):
    # On supprime entièrement les URLs (comme dans l'exemple de l'énoncé)
    return [combined_url_pattern.sub(' ', message) for message in text_list]

# Application
text = remove_urls(text)

In [16]:
print(text)

["upgrdcentre orange customer, you may now claim your free camera phone upgrade for your loyalty. call now on 0207 153 9153. offer ends 26th july. t&c's apply. opt-out available", "loan for any purpose £500 - £75,000. homeowners + tenants welcome. have you been previously refused? we can still help. call free 0800 1956669 or text back 'help'", 'contrast! nokia 3650 video camera phone is your call 09066382422 calls cost 150ppm ave call 3mins vary from mobiles 16+ close 300603 post bcm4284 lon wc1n3xx', 'urgent! your mobile number has been awarded with a £2000 prize guaranteed. call 09058094455 from land line. claim 3030. valid 12hrs only', 'someone has contacted our dating service and entered your phone because they fancy you! to find out who it is call from a landing 09111032124 . box12n146tf150p', 'send a logo 2 you are lover - 2 names joined by a heart. txt love name1 name2 mono eg love adam eve 07123456789 to 87077 yahoo! box36504w45wq xeno 4 no ads 150p', 'free entry into our £250 

In [17]:
# Removing numbers / digits

def remove_numbers(text_list):
    # On supprime les chiffres (les lettres collées restent : "26th" -> "th")
    return [re.sub(r'\d+', '', message) for message in text_list]

In [18]:
text = remove_numbers(text)

In [19]:
# Removing Punctuation & Special Characters
import string

def remove_punctuation(text_list):
    # On garde uniquement les lettres et les espaces.
    # Ainsi "t&c's" -> "tcs", "opt-out" -> "optout" (comme dans l'exemple)
    return [re.sub(r'[^a-z\s]', '', message) for message in text_list]

text = remove_punctuation(text)


# Handling double / multiple whitespace
def remove_extra_whitespace(text_list):
    # On remplace toute suite d'espaces par un seul, puis on enlève les bords.
    return [re.sub(r'\s+', ' ', message).strip() for message in text_list]

text = remove_extra_whitespace(text)

# Le texte est maintenant entièrement nettoyé -> on sauvegarde la colonne "Noise Removal"
df["Noise Removal"] = list(text)
df[["Case Normalization", "Noise Removal"]].head()

,Case Normalization,Noise Removal
0,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...
1,"loan for any purpose £500 - £75,000. homeowner...",loan for any purpose homeowners tenants welcom...
2,contrast! nokia 3650 video camera phone is you...,contrast nokia video camera phone is your call...
3,urgent! your mobile number has been awarded wi...,urgent your mobile number has been awarded wit...
4,someone has contacted our dating service and e...,someone has contacted our dating service and e...


# TOKENIZATION

In [20]:
# Tokenization : on découpe chaque message nettoyé en une liste de mots (tokens)
from nltk.tokenize import word_tokenize

# Les versions récentes de NLTK utilisent le modèle "punkt_tab"
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab")

def tokenize(text_list):
    return [word_tokenize(message) for message in text_list]

tokens = tokenize(text)

# On sauvegarde la liste de tokens dans la colonne "Tokenization"
df["Tokenization"] = tokens
print(tokens[0])

['upgrdcentre', 'orange', 'customer', 'you', 'may', 'now', 'claim', 'your', 'free', 'camera', 'phone', 'upgrade', 'for', 'your', 'loyalty', 'call', 'now', 'on', 'offer', 'ends', 'th', 'july', 'tcs', 'apply', 'optout', 'available']


# STOPWORDS REMOVAL

In [21]:
# Stopwords removal : on retire les mots vides (the, you, on, for, ...)
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(token_list):
    return [[w for w in toks if w not in stop_words] for toks in token_list]

tokens_no_stop = remove_stopwords(tokens)

df["Stopwords"] = tokens_no_stop
print(tokens_no_stop[0])

['upgrdcentre', 'orange', 'customer', 'may', 'claim', 'free', 'camera', 'phone', 'upgrade', 'loyalty', 'call', 'offer', 'ends', 'th', 'july', 'tcs', 'apply', 'optout', 'available']


# STEMMING & LEMMATIZATION

In [22]:
# Stemming (Porter) : on coupe les suffixes pour obtenir la racine (forme brute)
# Ex : "loyalty" -> "loyalti", "apply" -> "appli". On part de la colonne "Stopwords".
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()

def stem_tokens(token_list):
    return [[stemmer.stem(w) for w in toks] for toks in token_list]

stems = stem_tokens(tokens_no_stop)

df["Stemming"] = stems
print(stems[0])

['upgrdcentr', 'orang', 'custom', 'may', 'claim', 'free', 'camera', 'phone', 'upgrad', 'loyalti', 'call', 'offer', 'end', 'th', 'juli', 'tc', 'appli', 'optout', 'avail']


In [23]:
# Lemmatization (spaCy) : on ramène chaque mot à sa forme de base / dictionnaire
# Ex : "ends" -> "end". On part aussi de la colonne "Stopwords".
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize_tokens(token_list):
    lemmas = []
    for toks in token_list:
        # On reconstruit une phrase pour donner du contexte au lemmatiseur,
        # puis on récupère le lemme de chaque token.
        doc = nlp(" ".join(toks))
        lemmas.append([token.lemma_ for token in doc])
    return lemmas

lemmas = lemmatize_tokens(tokens_no_stop)

df["Lemmatization"] = lemmas
print(lemmas[0])

['upgrdcentre', 'orange', 'customer', 'may', 'claim', 'free', 'camera', 'phone', 'upgrade', 'loyalty', 'call', 'offer', 'end', 'th', 'july', 'tcs', 'apply', 'optout', 'available']


# Construction du DataFrame final (9 colonnes) & sauvegarde

In [24]:
# --- Construction du DataFrame final : 9 colonnes ---
# On remet les colonnes dans l'ordre demandé par l'énoncé.
final_cols = [
    "S. No.", "Message_body", "Label",
    "Case Normalization", "Noise Removal",
    "Tokenization", "Stopwords", "Stemming", "Lemmatization",
]
df = df[final_cols]

# Sauvegarde du dataset prétraité (9 colonnes)
df.to_csv("SMS_test_preprocessed.csv", index=False)

print("Dimensions finales :", df.shape, "->", df.shape[1], "colonnes")
print("Colonnes :", list(df.columns))
df.head()

Dimensions finales : (125, 9) -> 9 colonnes
Colonnes : ['S. No.', 'Message_body', 'Label', 'Case Normalization', 'Noise Removal', 'Tokenization', 'Stopwords', 'Stemming', 'Lemmatization']


,S. No.,Message_body,Label,Case Normalization,Noise Removal,Tokenization,Stopwords,Stemming,Lemmatization
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...,"[upgrdcentre, orange, customer, you, may, now,...","[upgrdcentre, orange, customer, may, claim, fr...","[upgrdcentr, orang, custom, may, claim, free, ...","[upgrdcentre, orange, customer, may, claim, fr..."
1,2,"Loan for any purpose £500 - £75,000. Homeowner...",Spam,"loan for any purpose £500 - £75,000. homeowner...",loan for any purpose homeowners tenants welcom...,"[loan, for, any, purpose, homeowners, tenants,...","[loan, purpose, homeowners, tenants, welcome, ...","[loan, purpos, homeown, tenant, welcom, previo...","[loan, purpose, homeowner, tenant, welcome, pr..."
2,3,Congrats! Nokia 3650 video camera phone is you...,Spam,contrast! nokia 3650 video camera phone is you...,contrast nokia video camera phone is your call...,"[contrast, nokia, video, camera, phone, is, yo...","[contrast, nokia, video, camera, phone, call, ...","[contrast, nokia, video, camera, phone, call, ...","[contrast, nokia, video, camera, phone, call, ..."
3,4,URGENT! Your Mobile number has been awarded wi...,Spam,urgent! your mobile number has been awarded wi...,urgent your mobile number has been awarded wit...,"[urgent, your, mobile, number, has, been, awar...","[urgent, mobile, number, awarded, prize, guara...","[urgent, mobil, number, award, prize, guarante...","[urgent, mobile, number, award, prize, guarant..."
4,5,Someone has contacted our dating service and e...,Spam,someone has contacted our dating service and e...,someone has contacted our dating service and e...,"[someone, has, contacted, our, dating, service...","[someone, contacted, dating, service, entered,...","[someon, contact, date, servic, enter, phone, ...","[someone, contact, date, service, enter, phone..."
